# SEC Form 13F table schema and structural audit

This notebook audits the seven Form 13F tables extracted from every configured SEC archive.

It answers four questions:

1. Is every expected table and metadata file present?
2. Do the Parquet columns agree with the SEC definition supplied with each archive?
3. Are SEC-required fields populated?
4. Are declared primary keys and parent–child accession links structurally consistent?

The notebook reads the interim Parquet files only. Archive hashes are verified during download, and filing selection and portfolio reconciliation are performed later in the pipeline.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_paths_config, get_sec_13f_config


paths = get_paths_config()
source = get_sec_13f_config()
tables_directory = paths.interim / "sec_13f" / "tables"

table_names = (
    "submission",
    "coverpage",
    "summarypage",
    "infotable",
    "othermanager",
    "othermanager2",
    "signature",
)

archive_ids = [archive.archive_id for archive in source.archives]

print(f"Project root: {project_root}")
print(f"Configured archives: {len(archive_ids):,}")
print(f"Expected tables per archive: {len(table_names)}")

Project root: C:\Users\salio\OneDrive\Desktop\Master Thesis
Configured archives: 53
Expected tables per archive: 7


## 2. Extracted-table inventory

The inventory uses Parquet metadata, so it can count rows and columns without loading the full tables into memory.

In [2]:
def load_table_definitions(metadata_path):
    with metadata_path.open("r", encoding="utf-8-sig") as file:
        metadata = json.load(file)

    return {
        Path(definition["url"]).stem.lower(): definition
        for definition in metadata["tables"]
    }


inventory_records = []
definitions_by_archive = {}
missing_outputs = []

for archive_id in archive_ids:
    archive_directory = tables_directory / archive_id
    metadata_path = archive_directory / "FORM13F_metadata.json"

    if not metadata_path.is_file():
        missing_outputs.append(metadata_path)
        continue

    definitions = load_table_definitions(metadata_path)
    definitions_by_archive[archive_id] = definitions

    for table_name in table_names:
        table_path = archive_directory / f"{table_name}.parquet"

        if not table_path.is_file():
            missing_outputs.append(table_path)
            continue

        parquet_file = pq.ParquetFile(table_path)
        arrow_schema = parquet_file.schema_arrow

        inventory_records.append(
            {
                "archive_id": archive_id,
                "table": table_name,
                "path": table_path,
                "rows": parquet_file.metadata.num_rows,
                "columns": len(arrow_schema.names),
                "column_names": tuple(arrow_schema.names),
                "all_columns_are_strings": all(
                    pa.types.is_string(field.type)
                    or pa.types.is_large_string(field.type)
                    for field in arrow_schema
                ),
            }
        )

if missing_outputs:
    missing_list = "\n".join(f"  - {path}" for path in missing_outputs)
    raise FileNotFoundError(
        f"Required extracted outputs are missing:\n{missing_list}"
    )

inventory = pd.DataFrame(inventory_records)

inventory_summary = (
    inventory.groupby("table", as_index=False)
    .agg(
        archives=("archive_id", "nunique"),
        total_rows=("rows", "sum"),
        min_columns=("columns", "min"),
        max_columns=("columns", "max"),
    )
    .sort_values("table")
)

display(inventory_summary.style.format({"total_rows": "{:,}"}))

print()
print(f"Expected Parquet files: {len(archive_ids) * len(table_names):,}")
print(f"Found Parquet files: {len(inventory):,}")
print(f"Found metadata files: {len(definitions_by_archive):,}")

,table,archives,total_rows,min_columns,max_columns
0,coverpage,53,"397,833",21,21
1,infotable,53,"120,182,194",15,15
2,othermanager,53,"202,203",7,7
3,othermanager2,53,"159,211",7,7
4,signature,53,"397,833",8,8
5,submission,53,"397,833",5,5
6,summarypage,53,"316,090",5,5



Expected Parquet files: 371
Found Parquet files: 371
Found metadata files: 53


## 3. Schema agreement

SEC schemas may evolve over time. Therefore, each Parquet table is compared with the metadata from its own archive rather than with one universal schema.

In [3]:
schema_records = []

for row in inventory.itertuples(index=False):
    definition = definitions_by_archive[row.archive_id][row.table]
    expected_columns = tuple(
        column["name"]
        for column in definition["tableSchema"]["columns"]
    )

    schema_records.append(
        {
            "archive_id": row.archive_id,
            "table": row.table,
            "columns_match_metadata": row.column_names == expected_columns,
            "parquet_only_columns": sorted(
                set(row.column_names) - set(expected_columns)
            ),
            "metadata_only_columns": sorted(
                set(expected_columns) - set(row.column_names)
            ),
            "all_columns_are_strings": row.all_columns_are_strings,
            "schema_signature": row.column_names,
        }
    )

schema_audit = pd.DataFrame(schema_records)

schema_summary_records = []

for table_name, group in schema_audit.groupby("table", sort=True):
    schema_summary_records.append(
        {
            "table": table_name,
            "archives": group["archive_id"].nunique(),
            "schema_versions": group["schema_signature"].nunique(),
            "metadata_matches": int(group["columns_match_metadata"].sum()),
            "string_storage_matches": int(group["all_columns_are_strings"].sum()),
        }
    )

schema_summary = pd.DataFrame(schema_summary_records)
display(schema_summary)

print()
print(
    "Parquet/metadata mismatches: "
    f"{(~schema_audit['columns_match_metadata']).sum():,}"
)
print(
    "Files with a non-string stored column: "
    f"{(~schema_audit['all_columns_are_strings']).sum():,}"
)

schema_mismatches = schema_audit.loc[
    ~schema_audit["columns_match_metadata"],
    [
        "archive_id",
        "table",
        "parquet_only_columns",
        "metadata_only_columns",
    ],
]

if not schema_mismatches.empty:
    print()
    print("Mismatches to review")
    display(schema_mismatches.reset_index(drop=True))

,table,archives,schema_versions,metadata_matches,string_storage_matches
0,coverpage,53,1,52,53
1,infotable,53,1,52,53
2,othermanager,53,1,52,53
3,othermanager2,53,1,52,53
4,signature,53,1,53,53
5,submission,53,1,52,53
6,summarypage,53,1,53,53



Parquet/metadata mismatches: 5
Files with a non-string stored column: 0

Mismatches to review


,archive_id,table,parquet_only_columns,metadata_only_columns
0,2022_q1,submission,[],[FILENUMBER]
1,2022_q1,coverpage,"[CRDNUMBER, SECFILENUMBER]",[]
2,2022_q1,infotable,[FIGI],[]
3,2022_q1,othermanager,"[CRDNUMBER, SECFILENUMBER]",[]
4,2022_q1,othermanager2,"[CRDNUMBER, SECFILENUMBER]",[]


### Current SEC data dictionary

The most recent configured archive is used to present the current variable definitions. Earlier schema versions remain visible in the schema-version counts above.

In [4]:
current_archive_id = archive_ids[0]
current_definitions = definitions_by_archive[current_archive_id]

dictionary_records = []

for table_name in table_names:
    table_schema = current_definitions[table_name]["tableSchema"]
    primary_key = table_schema.get("primaryKey", [])

    if isinstance(primary_key, str):
        primary_key = [primary_key]

    for column in table_schema["columns"]:
        datatype = column.get("datatype", {})
        datatype_name = (
            datatype.get("base", "")
            if isinstance(datatype, dict)
            else str(datatype)
        )

        dictionary_records.append(
            {
                "table": table_name,
                "column": column["name"],
                "type": datatype_name,
                "required": bool(column.get("required", False)),
                "primary_key": column["name"] in primary_key,
                "description": column.get("dc:description", ""),
            }
        )

data_dictionary = pd.DataFrame(dictionary_records)

print(f"Data dictionary from archive: {current_archive_id}")

for table_name in table_names:
    print()
    print(table_name.upper())
    display(
        data_dictionary.loc[
            data_dictionary["table"].eq(table_name),
            ["column", "type", "required", "primary_key", "description"],
        ].reset_index(drop=True)
    )

Data dictionary from archive: 2026_mar_may

SUBMISSION


,column,type,required,primary_key,description
0,ACCESSION_NUMBER,string,True,True,Unique identifier assigned by the SEC to each ...
1,FILING_DATE,date,True,False,Filing date with the Commission; sourced from ...
2,SUBMISSIONTYPE,string,True,False,Submission type is either an initial quarterly...
3,CIK,string,True,False,Filer CIK (Central index key)
4,PERIODOFREPORT,date,True,False,Period Of Report in (DD-MON-YYYY) format



COVERPAGE


,column,type,required,primary_key,description
0,ACCESSION_NUMBER,string,True,True,Unique identifier assigned by the SEC to each ...
1,REPORTCALENDARORQUARTER,date,True,False,Report for the calendar year of quater ended i...
2,ISAMENDMENT,string,False,False,Check here if amendment
3,AMENDMENTNO,NUMBER,False,False,Amendment number
4,AMENDMENTTYPE,string,False,False,Amendment type is a restatement or adds new ho...
5,CONFDENIEDEXPIRED,string,False,False,Whether confidential treatment request is deni...
6,DATEDENIEDEXPIRED,date,False,False,Pursuant to a request for confidential tretmen...
7,DATEREPORTED,date,False,False,Date securities holdings reported pursuant to ...
8,REASONFORNONCONFIDENTIALITY,string,False,False,Reason for non-confidentiality: denied or conf...
9,FILINGMANAGER_NAME,string,True,False,Filing manager name



SUMMARYPAGE


...

,column,type,required,primary_key,description
0,ACCESSION_NUMBER,string,True,True,Unique identifier assigned by the SEC to each ...
1,OTHERMANAGER_SK,NUMBER,True,True,Other manager surrogate key
2,CIK,string,False,False,Central index key
3,FORM13FFILENUMBER,string,False,False,FORM 13F File Number
4,CRDNUMBER,string,False,False,CRD Number (if applicable)
5,SECFILENUMBER,string,False,False,SEC File Number (if applicable)
6,NAME,string,True,False,Other Manager Name



OTHERMANAGER2


,column,type,required,primary_key,description
0,ACCESSION_NUMBER,string,True,True,Unique identifier assigned by the SEC to each ...
1,SEQUENCENUMBER,NUMBER,True,False,Sequence number
2,CIK,string,False,False,Central index key
3,FORM13FFILENUMBER,string,False,False,FORM 13F FILE NUMBER
4,CRDNUMBER,string,False,False,CRD Number (if applicable)
5,SECFILENUMBER,string,False,False,SEC File Number (if applicable)
6,NAME,string,True,False,Other manager2 Name



SIGNATURE


,column,type,required,primary_key,description
0,ACCESSION_NUMBER,string,True,True,Unique identifier assigned by the SEC to each ...
1,NAME,string,True,False,Signatory name
2,TITLE,string,True,False,Signatory title
3,PHONE,string,False,False,Signatory phone
4,SIGNATURE,string,True,False,Signatory signature
5,CITY,string,True,False,Signatory City name
6,STATEORCOUNTRY,string,True,False,Signatory state or country
7,SIGNATUREDATE,date,True,False,Signature date in (DD-MON-YYYY) format


## 4. Required-field completeness

Blank source fields were converted to missing values during extraction. Null counts are taken from Parquet metadata where available, avoiding a full-table load.

In [5]:
def count_nulls(parquet_file, column_name):
    column_index = parquet_file.schema_arrow.names.index(column_name)
    null_counts = []

    for row_group_index in range(parquet_file.num_row_groups):
        statistics = parquet_file.metadata.row_group(row_group_index).column(
            column_index
        ).statistics

        if statistics is None or statistics.null_count is None:
            table = parquet_file.read(columns=[column_name])
            return table[column_name].null_count

        null_counts.append(statistics.null_count)

    return sum(null_counts)


required_records = []

for row in inventory.itertuples(index=False):
    definition = definitions_by_archive[row.archive_id][row.table]
    required_columns = [
        column["name"]
        for column in definition["tableSchema"]["columns"]
        if column.get("required", False)
    ]
    parquet_file = pq.ParquetFile(row.path)

    for column_name in required_columns:
        required_records.append(
            {
                "archive_id": row.archive_id,
                "table": row.table,
                "column": column_name,
                "null_values": count_nulls(parquet_file, column_name),
            }
        )

required_audit = pd.DataFrame(required_records)

required_summary_records = []

for table_name, group in required_audit.groupby("table", sort=True):
    required_summary_records.append(
        {
            "table": table_name,
            "archive_field_checks": len(group),
            "null_values": int(group["null_values"].sum()),
            "failing_archive_fields": int(group["null_values"].gt(0).sum()),
        }
    )

required_summary = pd.DataFrame(required_summary_records)
display(required_summary.style.format({"null_values": "{:,}"}))

print()
print(f"Required-field null values: {required_audit['null_values'].sum():,}")

required_failures = required_audit.loc[
    required_audit["null_values"].gt(0)
]

if not required_failures.empty:
    print()
    print("Required fields with null values")
    display(required_failures.reset_index(drop=True))

,table,archive_field_checks,null_values,failing_archive_fields
0,coverpage,265,0,0
1,infotable,636,2,2
2,othermanager,159,0,0
3,othermanager2,159,0,0
4,signature,371,0,0
5,submission,265,0,0
6,summarypage,53,0,0



Required-field null values: 2

Required fields with null values


,archive_id,table,column,null_values
0,2025_sep_nov,infotable,NAMEOFISSUER,1
1,2025_jun_aug,infotable,NAMEOFISSUER,1


## 5. Declared keys and parent–child links

Primary keys are taken directly from each archive's SEC metadata. Key checks are performed within each archive. Every non-`SUBMISSION` table is also checked for accession numbers that do not resolve to a submission in the same archive.

In [6]:
def quote_identifier(name):
    return f'"{name.replace(chr(34), chr(34) * 2)}"'


connection = duckdb.connect(database=":memory:")
key_records = []
relationship_records = []

for row in inventory.itertuples(index=False):
    table_schema = definitions_by_archive[row.archive_id][row.table][
        "tableSchema"
    ]
    primary_key = table_schema.get("primaryKey", [])

    if isinstance(primary_key, str):
        primary_key = [primary_key]

    quoted_keys = [quote_identifier(column) for column in primary_key]
    key_expression = (
        quoted_keys[0]
        if len(quoted_keys) == 1
        else f"({', '.join(quoted_keys)})"
    )
    complete_key = " AND ".join(
        f"{column} IS NOT NULL" for column in quoted_keys
    )

    row_count, null_key_rows, unique_valid_keys = connection.execute(
        f''' 
        SELECT
            COUNT(*),
            COALESCE(
                SUM(CASE WHEN NOT ({complete_key}) THEN 1 ELSE 0 END),
                0
            ),
            COUNT(DISTINCT {key_expression})
                FILTER (WHERE {complete_key})
        FROM read_parquet(?)
        ''',
        [str(row.path)],
    ).fetchone()

    key_records.append(
        {
            "archive_id": row.archive_id,
            "table": row.table,
            "declared_key": " + ".join(primary_key),
            "null_key_rows": int(null_key_rows),
            "duplicate_rows": int(
                row_count - null_key_rows - unique_valid_keys
            ),
        }
    )

    if row.table != "submission":
        submission_path = (
            tables_directory / row.archive_id / "submission.parquet"
        )
        orphan_rows = connection.execute(
            '''
            SELECT COUNT(*)
            FROM read_parquet(?) AS child
            LEFT JOIN read_parquet(?) AS parent
                ON child.ACCESSION_NUMBER = parent.ACCESSION_NUMBER
            WHERE parent.ACCESSION_NUMBER IS NULL
            ''',
            [str(row.path), str(submission_path)],
        ).fetchone()[0]

        relationship_records.append(
            {
                "archive_id": row.archive_id,
                "table": row.table,
                "orphan_rows": int(orphan_rows),
            }
        )

othermanager2_candidate_duplicates = 0
othermanager2_exact_duplicates = 0

for archive_id in archive_ids:
    table_path = tables_directory / archive_id / "othermanager2.parquet"
    row_count, unique_candidate_keys, unique_rows = connection.execute(
        '''
        SELECT
            COUNT(*),
            COUNT(DISTINCT (ACCESSION_NUMBER, SEQUENCENUMBER)),
            COUNT(DISTINCT (
                ACCESSION_NUMBER,
                SEQUENCENUMBER,
                CIK,
                FORM13FFILENUMBER,
                CRDNUMBER,
                SECFILENUMBER,
                NAME
            ))
        FROM read_parquet(?)
        ''',
        [str(table_path)],
    ).fetchone()
    othermanager2_candidate_duplicates += row_count - unique_candidate_keys
    othermanager2_exact_duplicates += row_count - unique_rows

connection.close()

key_audit = pd.DataFrame(key_records)
relationship_audit = pd.DataFrame(relationship_records)

key_summary_records = []

for table_name, group in key_audit.groupby("table", sort=True):
    key_summary_records.append(
        {
            "table": table_name,
            "declared_key": "; ".join(sorted(group["declared_key"].unique())),
            "null_key_rows": int(group["null_key_rows"].sum()),
            "duplicate_rows": int(group["duplicate_rows"].sum()),
            "failing_archives": int(
                (group["null_key_rows"].gt(0) | group["duplicate_rows"].gt(0)).sum()
            ),
        }
    )

key_summary = pd.DataFrame(key_summary_records)

relationship_summary = (
    relationship_audit.groupby("table", as_index=False)
    .agg(
        orphan_rows=("orphan_rows", "sum"),
        failing_archives=("orphan_rows", lambda values: values.gt(0).sum()),
    )
    .sort_values("table")
)

print("SEC-declared primary keys")
display(key_summary.style.format({"null_key_rows": "{:,}", "duplicate_rows": "{:,}"}))

print("Parent SUBMISSION links")
display(relationship_summary.style.format({"orphan_rows": "{:,}"}))

print(
    "Duplicate rows under the candidate OTHERMANAGER2 key "
    "(ACCESSION_NUMBER + SEQUENCENUMBER): "
    f"{othermanager2_candidate_duplicates:,}"
)
print(
    "Exact duplicate OTHERMANAGER2 rows: "
    f"{othermanager2_exact_duplicates:,}"
)

SEC-declared primary keys


,table,declared_key,null_key_rows,duplicate_rows,failing_archives
0,coverpage,ACCESSION_NUMBER,0,0,0
1,infotable,ACCESSION_NUMBER + INFOTABLE_SK,0,0,0
2,othermanager,ACCESSION_NUMBER + OTHERMANAGER_SK,0,0,0
3,othermanager2,ACCESSION_NUMBER,0,"106,909",53
4,signature,ACCESSION_NUMBER,0,0,0
5,submission,ACCESSION_NUMBER,0,0,0
6,summarypage,ACCESSION_NUMBER,0,0,0


Parent SUBMISSION links


,table,orphan_rows,failing_archives
0,coverpage,0,0
1,infotable,0,0
2,othermanager,0,0
3,othermanager2,0,0
4,signature,0,0
5,summarypage,0,0


Duplicate rows under the candidate OTHERMANAGER2 key (ACCESSION_NUMBER + SEQUENCENUMBER): 606
Exact duplicate OTHERMANAGER2 rows: 0


## 6. Interpretation

- **Coverage is complete.** All 371 expected Parquet files and all 53 SEC metadata files are present. The tables contain 397,833 submissions and 120,182,194 reported holding rows before later filing selection.
- **The extracted schemas are stable.** Each table has one actual Parquet schema across the 53 archives, and every stored column is a string as intended at the extraction stage.
- **The 2022 Q1 SEC package contains a metadata anomaly.** Five tables differ from that archive's JSON definition even though their Parquet columns match the bundled TSV headers verified by Script 02. The JSON omits columns present in four TSVs and lists `FILENUMBER` in `SUBMISSION` although that column is absent from the TSV. This is a source-metadata inconsistency, not an extraction-schema failure.
- **Required-field completeness is nearly perfect.** Two `INFOTABLE` rows have a missing `NAMEOFISSUER`, one in `2025_jun_aug` and one in `2025_sep_nov`. These source-level missing values should be retained and handled explicitly if those rows reach the analytical sample.
- **Keys and table links are usable with one limitation.** All SEC-declared keys are complete and unique except the declared `OTHERMANAGER2` key. In that table, `ACCESSION_NUMBER` alone produces 106,909 duplicate rows because a filing may list several managers. Adding `SEQUENCENUMBER` reduces the excess to 606 but does not create a unique row key; those records differ in other manager fields, and there are no exact duplicate rows. Consequently, `OTHERMANAGER2` should not be joined as though either combination were unique. No child-table row has an accession number missing from `SUBMISSION`.

Overall, the extracted layer is structurally suitable for the filing-selection stage. The audit identifies two limited SEC source issues that should be documented rather than silently altered.